# Chakra RUN_F: site-by-site diagnostics (no training)

**Why.** RUN_E (30 Sep) found that the ViT-L trained with 20 % mixed polyp-free frames **misses 56 % of PolypGen C4
(Oslo) polyps**, against 16 % without them, and flags only 3 % of C4 normal frames, against 80 %.

**What this notebook decides.** Every attached run is scored site by site, then the question is settled:
- **H1:** PolypGen negative videos from C4 in training (with no PolypGen polyps) teach the model "C4 look = no polyp";
- **H0:** any polyp-free training frames make the model conservative, and C4 is just the hardest site.

The R6 CNN arms separate the two: HyperKvasir-only, PolypGen-only and mixed. Polyp frames now also record peak
probability, so the notebook also measures how many silent misses a lower threshold would recover.

**Settings:** GPU T4 x2, Internet ON.

**Inputs:**
- `chakra_vit_fix` **v1.2** (this package);
- `chakra-leakbench-v1`, `chakra-leakbench-e4-v1`, `chakra-leakbench-s2-v1`;
- every notebook output or dataset holding `runs/<tag>/best.pt`: the R6 CNN runs, E4, the E1 checkpoints and RUN_A/B/C.

Run cell 2 first and read its list. Then **Save Version → Save & Run All**. About 1–2 h (estimate).
Download `SEND_BACK_SITE.zip` from the Output tab.

In [ ]:
# ---- 1. bundle, GPUs, options -------------------------------------------------------------------------
import os, sys, glob, json, subprocess, time
SMOKE = os.environ.get("CHAKRA_SMOKE") == "1"          # developer switch (CPU, tiny data); leave off
ROOTS = [r for r in os.environ.get("CHAKRA_ROOTS", "/kaggle/input").split(os.pathsep) if r]
hits = sorted(p for r in ROOTS for p in glob.glob(os.path.join(r, "**", "CHAKRA_BUNDLE_ID"), recursive=True))
hits = [h for h in hits if "v1.2" in open(h).read()] or hits
assert hits, "bundle not found: add the uploaded chakra_vit_fix dataset (v1.2) as an Input"
BUNDLE = os.path.dirname(hits[0]); sys.path.insert(0, BUNDLE)
print("bundle:", BUNDLE, "|", open(hits[0]).read().strip())
assert "v1.2" in open(hits[0]).read(), "this notebook needs chakra_vit_fix v1.2 (upload the new zip as a new version)"
import torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "NONE")
if N_GPU == 0 and not SMOKE:
    raise SystemExit("No GPU: Settings -> Accelerator -> GPU T4 x2")
W = os.environ.get("CHAKRA_WORK") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())
OUT = os.path.join(W, "site_diag")
INCLUDE = ""          # regex on run tags, e.g. r"r6_|e4|v2__(xattn|unet)" = the decisive CNN runs only
EXCLUDE = ""
MAX_MINUTES = 600     # stop starting new jobs after 10 h (Kaggle sessions end at 12 h); re-run resumes
import site_diag
ARGS = ["--roots", *ROOTS, "--out", OUT, "--max-minutes", str(MAX_MINUTES)] + \
       (["--include", INCLUDE] if INCLUDE else []) + (["--exclude", EXCLUDE] if EXCLUDE else []) + \
       (["--gpus", "0", "--workers", "0"] if SMOKE else [])

In [ ]:
# ---- 2. what will be scored (nothing runs yet): runs, arms, data, xattn code, missing decisive arms ----------------
site_diag.main(ARGS + ["--dry-run"])

In [ ]:
# ---- 3. score every run at its training size, then the site report (finished runs are skipped on a re-run) --------
t0 = time.time()
decision = site_diag.main(ARGS)
print(f"\n{(time.time() - t0) / 60:.0f} min")
print(json.dumps(decision, indent=1))

In [ ]:
# ---- 4. the report (also inside SEND_BACK_SITE.zip) ---------------------------------------------------------------
print(open(os.path.join(OUT, "SITE_REPORT.md"), encoding="utf-8").read())
print("send back:", os.path.join(OUT, "SEND_BACK_SITE.zip"))

**Download** `site_diag/SEND_BACK_SITE.zip` from the Output tab and send it back.